In [6]:
#Environment Setup
!pip install -q transformers sentence-transformers scikit-learn datasets tqdm

In [7]:
#Text Classification with a Task-Specific Model
#The fastest way to perform classification is using a fine-tuned representation pipeline.
from transformers import pipeline

# Load a dedicated sentiment analysis pipeline (DistilBERT base model)
classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

reviews = [
    "This movie was an absolute masterpiece with stunning visual effects!",
    "The plot was boring, predictable, and a total waste of time."
]

results = classifier(reviews)

for text, res in zip(reviews, results):
    print(f"Review: '{text}'")
    print(f"Prediction: {res['label']} (Confidence: {res['score']:.4f})\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Review: 'This movie was an absolute masterpiece with stunning visual effects!'
Prediction: POSITIVE (Confidence: 0.9999)

Review: 'The plot was boring, predictable, and a total waste of time.'
Prediction: NEGATIVE (Confidence: 0.9998)



In [8]:
#Classification via Document Embeddings + Logistic Regression
#Instead of end-to-end models, we can extract dense document representations and use a lightweight linear classifier
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Load embedding representation model
embedder = SentenceTransformer("thenlper/gte-small")

# 2. Sample Training Set (Text & Labels)
train_texts = [
    "I loved the film, amazing acting!",
    "Great movie, highly recommended.",
    "Terrible experience, poorly directed.",
    "Horrible dialogue and awful story."
]
train_labels = [1, 1, 0, 0] # 1: Positive, 0: Negative

# 3. Sample Test Set
test_texts = [
    "An excellent piece of cinema.",
    "Boring and unpleasant to watch."
]
test_labels = [1, 0]

# 4. Generate Embeddings
X_train = embedder.encode(train_texts)
X_test = embedder.encode(test_texts)

# 5. Train a Logistic Regression Classifier on the static embeddings
clf = LogisticRegression()
clf.fit(X_train, train_labels)

# 6. Evaluate
predictions = clf.predict(X_test)
print("--- EMBEDDING CLASSIFICATION RESULTS ---")
for text, pred in zip(test_texts, predictions):
    label_str = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"Text: '{text}' -> Predicted: {label_str}")

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 66.7MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- EMBEDDING CLASSIFICATION RESULTS ---
Text: 'An excellent piece of cinema.' -> Predicted: POSITIVE
Text: 'Boring and unpleasant to watch.' -> Predicted: NEGATIVE


In [12]:
#Zero-Shot / Generative Classification (Flan-T5)
#Using an open-source generative model (Flan-T5) in a text-to-text setup to perform zero-shot classification without dataset training.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load a text-to-text generation model (Flan-T5-base)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Construct a zero-shot instruction prompt
prompt_template = "Is this review positive or negative? Review: {review}"

test_review = "Unpretentious, charming, original, and deeply moving."

prompt = prompt_template.format(review=test_review)

input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model.generate(input_ids, max_new_tokens=2) # Generate only 'positive' or 'negative'
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- GENERATIVE CLASSIFICATION ---")
print("Prompt:\n", prompt)
print("Generated Label:", generated_text.strip())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- GENERATIVE CLASSIFICATION ---
Prompt:
 Is this review positive or negative? Review: Unpretentious, charming, original, and deeply moving.
Generated Label: positive
